# MobiHet V2 A — feasibility
All learned models DAGSNet. No local raw/perturbed record shared. Internal group/old only. Protocol and gates frozen in input pilot-v2.md. R20 headline; R2 design only. Unlimited contact stage.

Ten metrics per client confusion: accuracy=TP/N; class precision=TP/(TP+FP), recall=TP/(TP+FN), F1=2PR/(P+R); macro=class mean, weighted=support-weighted mean, micro=pooled counts. Single-label accuracy=precision_micro=recall_micro=recall_weighted=f1_micro. Private headline gp; teacher-only B headline head.

W&B online before training; credential preferred attached secret, private inline fallback owner-authorized.

In [1]:

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.1 which is incompatible.
wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] 

W&B authenticated online


In [2]:
CODE=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import torch
count=torch.cuda.device_count()
assert count==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(count))
assert torch.__version__=='2.10.0+cu128',torch.__version__
print('torch',torch.__version__,'gpus',[torch.cuda.get_device_name(i) for i in range(count)],flush=True)
hits=[p.parent for p in Path('/kaggle/input').rglob('v2_run.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0]; WORK=Path('/kaggle/working/work'); OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source'); SOURCE.mkdir(exist_ok=True)
for name in ('mh','knowledge','papers/pilot/fixture'): (WORK/name).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
for f in SRC.iterdir():
    if f.name in CODE: dst=WORK/'mh'/f.name
    elif f.name=='meta.json': dst=WORK/'knowledge'/f.name
    elif f.name=='pilot-v2.md': continue
    else: dst=WORK/'papers/pilot/fixture'/f.name
    shutil.copyfile(f,dst)
fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('fixture mismatch',name)
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
print('verified source and fixture',len(manifest),len(fixture['files']),flush=True)
sys.path.insert(0,str(WORK))
import numpy as np
from mh.dagsnet import build_model,CFG
from mh.v2_math import Layout,CountSketch,SeedBasis,unpack
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
for gpu in range(2):
    device=torch.device(f'cuda:{gpu}'); torch.cuda.set_device(device); torch.manual_seed(42)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(device).eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy')[:32].astype('float32')).to(device)
    y=torch.from_numpy(np.load(SRC/'train_0_y.npy')[:32].astype('int64')).to(device)
    layout=Layout(model); assert layout.size==395024
    with torch.autocast('cuda',dtype=torch.float16): z=model(x)[0].float()
    loss=torch.nn.functional.cross_entropy(z,y); assert torch.isfinite(loss); loss.backward()
    gradient=torch.cat([p.grad.float().reshape(-1) for p in model.parameters()])
    assert torch.isfinite(gradient).all()
    sketch=CountSketch(layout.size,3,256,1701,device)
    packet,delta=sketch.patch(sketch.encode(gradient),128,0); assert torch.isfinite(delta).all()
    assert unpack(packet,device,0)['version']==1
    basis=SeedBasis(model,8,1901,device); coeff=torch.zeros(8,device=device,requires_grad=True)
    torch.nn.functional.cross_entropy(basis.forward(model,coeff,x)[0].float(),y).backward()
    assert torch.isfinite(coeff.grad).all()
    cm=torch.bincount(y*16+z.argmax(1),minlength=256); assert int(cm.sum())==len(y)
    print('T4 real-model deterministic preflight PASS',gpu,flush=True)
    del model,x,y,z,gradient,sketch,basis,coeff,delta; torch.cuda.empty_cache()
torch.cuda.set_device(0)


torch 2.10.0+cu128 gpus ['Tesla T4', 'Tesla T4']
verified source and fixture 16 66
T4 real-model deterministic preflight PASS 0
T4 real-model deterministic preflight PASS 1


In [3]:
SEED=42
METHODS=['local', 'a_own', 'a_dense', 'a_small_noef', 'a_small_ef', 'a_medium_noef', 'a_medium_ef']

T0=time.time(); OUT.mkdir(exist_ok=True)
queue=list(METHODS); slots=[0,0,1,1]; active={}; done={}; seen={}
while queue or active:
    for slot,gpu in enumerate(slots):
        if slot in active or not queue or time.time()-T0>=3*3600: continue
        arm=queue.pop(0); folder=OUT/f'{arm}_s{SEED}'; folder.mkdir(exist_ok=True)
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        command=[sys.executable,'-m','mh.v2_run','--arm',arm,'--seed',str(SEED),'--out',str(OUT),
                 '--wandb_online','--max_hours','4']
        log=open(folder/'log.txt','w')
        active[slot]=(arm,subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT),log)
        print('start',arm,'gpu',gpu,flush=True)
    for slot,(arm,proc,log) in list(active.items()):
        lines=(OUT/f'{arm}_s{SEED}'/'log.txt').read_text().splitlines()
        for line in lines[seen.get(arm,0):]:
            if line.startswith(('setup','r','done','WANDB','Traceback','budget')) or 'Error' in line:
                print(arm,'|',line,flush=True)
        seen[arm]=len(lines)
        if proc.poll() is not None:
            done[arm]=proc.returncode; log.close(); del active[slot]
    if queue and not active and time.time()-T0>=3*3600: break
    if queue or active: time.sleep(20)
summary={arm:{'rc':rc,'complete':json.loads((OUT/f'{arm}_s{SEED}'/'result.json').read_text())['complete']
             if (OUT/f'{arm}_s{SEED}'/'result.json').exists() else False} for arm,rc in done.items()}
(OUT/f'summary_s{SEED}.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary),flush=True)
shutil.rmtree(WORK)  # raw fixture never published as notebook output
needle=os.environ['WANDB_API_KEY'].encode()
for path in Path('/kaggle/working').rglob('*'):
    if not path.is_file(): continue
    carry=b''
    with path.open('rb') as handle:
        while True:
            part=handle.read(1024*1024)
            if not part: break
            assert needle not in carry+part,('credential in output',path.name)
            carry=(carry+part)[-len(needle):]
assert set(summary)==set(METHODS) and all(v['rc']==0 and v['complete'] for v in summary.values()),summary


start local gpu 0
start a_own gpu 0
start a_dense gpu 1
start a_small_noef gpu 1
local | WANDB https://wandb.ai/21522798-uit/mobihet-v2/runs/local_s42-bc06069d
local | setup local role private rows 1000369 sec 2.3
a_own | WANDB https://wandb.ai/21522798-uit/mobihet-v2/runs/a_own_s42-7cc3dcf8
a_own | setup a_own role private rows 1000369 sec 3.1
a_dense | WANDB https://wandb.ai/21522798-uit/mobihet-v2/runs/a_dense_s42-d3f39899
a_dense | setup a_dense role private rows 1000369 sec 3.1
a_small_noef | WANDB https://wandb.ai/21522798-uit/mobihet-v2/runs/a_small_noef_s42-2574da92
a_small_noef | setup a_small_noef role private rows 1000369 sec 3.2
local | r1 local s42 private group gp None loss 0.8078 79s
local | r2 local s42 private group gp None loss 0.4390 77s
a_own | r1 a_own s42 private group gp None loss 0.8078 162s
a_dense | r1 a_dense s42 private group gp None loss 0.8078 165s
a_small_noef | r1 a_small_noef s42 private group gp None loss 0.8078 167s
local | r3 local s42 private group 

AssertionError: ('credential in output', '__notebook__.ipynb')